# MCP on Databricks — UI walkthrough

This is the **point-and-click** version of `02_MCP on Databricks`. It walks the same journey — build a managed MCP tool, route an external weather MCP server through the **Unity AI Gateway**, and use it in an agent — but everything is done in the Databricks **console** instead of with the SDK/CLI.

Use whichever fits you: `02` if you like code, this one if you'd rather click. A couple of steps are the same in both (a Unity Catalog function is SQL either way, and deploying the app uses the CLI) — those are called out below.

By the end you'll have:
1. A managed MCP tool (a Unity Catalog function).
2. An external weather MCP server, hosted on a Databricks App, **registered through the Unity AI Gateway**.
3. That governed weather tool used in an agent — with permissions, tool filtering, and an audit trail.

## Prerequisites

- A Unity Catalog-enabled workspace with **serverless** compute, in a region where Model Serving is available.
- These previews enabled for your account (**Settings → Previews**, or ask an admin): **Unity AI Gateway** and **Managed MCP Servers**.
- Permission to create a connection and a service in a schema you can write to.

> The Unity AI Gateway and Managed MCP Servers are in **Beta**; the screens below may shift over time.

## The three kinds of MCP server

1. **Managed MCP** — Databricks-hosted servers for built-in features (Genie, Vector Search, DBSQL, Unity Catalog functions). No setup.
2. **External MCP** — a third-party server hosted outside Databricks, reached through a governed Unity Catalog connection. **This is the focus.**
3. **Custom MCP** — your own server, hosted as a Databricks App.

You'll see all three in one place later, in the agent's **Add tools → MCP Servers** panel.

## 1. Managed MCP: a Unity Catalog function

Any Unity Catalog function is callable by an agent through the managed `functions` MCP server. Creating a function is SQL in either version of this notebook — run this in a **SQL Editor** query or a notebook cell. (It uses the `cust_service_data` table from `00_Setup`.)

```sql
CREATE OR REPLACE FUNCTION main.default.get_order_history(user_name STRING)
RETURNS TABLE (returns_last_12_months INT, issue_category STRING, todays_date DATE)
COMMENT 'Returns the number of returns and issue category for a customer'
LANGUAGE SQL
RETURN
  SELECT COUNT(*) AS returns_last_12_months, issue_category, now() AS todays_date
  FROM main.default.cust_service_data
  WHERE name = user_name
  GROUP BY issue_category;
```

That function is now available to agents as a managed MCP tool. On to the main event.

## 2. External MCP through the Unity AI Gateway

We'll take a third-party weather MCP server (the one in [`weather-mcp-server/`](./weather-mcp-server), Open-Meteo, no API key), host it on a Databricks App, and route it through the gateway so every call is governed.

```
  Agent  ─►  Unity AI Gateway  ─►  UC HTTP connection  ─►  Weather MCP app  ─►  Open-Meteo
                    └── checks permission • filters tools • logs every call
```

### 2a. Host the server as a Databricks App

Deploying an app is a CLI step in both versions (there's a UI create-app flow too, but the CLI is simplest for source-backed apps). From a terminal at the repo root:

```bash
databricks apps create weather-mcp-server --description "External weather MCP server (Open-Meteo)"
databricks sync ./weather-mcp-server /Workspace/Users/<you>/weather-mcp-server
databricks apps deploy weather-mcp-server --source-code-path /Workspace/Users/<you>/weather-mcp-server
```

Get the app URL from **Compute → Apps → weather-mcp-server** (or `databricks apps get weather-mcp-server -o json | jq -r .url`). The MCP endpoint is `<app-url>/mcp`. See `weather-mcp-server/README.md` for why the server runs over HTTP.

### 2b. Create the Unity Catalog HTTP connection

This is the step the workspace nudges you toward with *"Use Unity Catalog connections for secure HTTP requests"* — it stores the app credential so nothing sensitive lives in your code.

1. Go to **Catalog → Connections → Create connection** (or **Catalog Explorer → External Data → Connections**).
2. **Connection name:** `weather_mcp_conn`. **Connection type:** `HTTP`. Click **Next**.

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_connection_form.png" width="70%">

3. The form expands. Turn on **Create under a schema** and pick a catalog + schema you can write to. Then set **Auth type**. Your options:

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_connection_auth_types.png" width="70%">

- **Bearer token** — simplest; store a Databricks token belonging to a principal that has **Can use** on the app. No admin needed.
- **OAuth Machine to Machine** — a service principal (client credentials). The durable, non-personal option, but creating the SP + secret needs workspace admin.
- The **OAuth User to Machine** variants and **Dynamic Client Registration** are for per-user / auto-provisioned auth.

4. Fill the remaining fields — **Host** = `https://<app-host>.databricksapps.com`, **Port** = `443`, **Base path** = `/mcp`, and the token/credentials for your chosen auth type. Create the connection.

The finished connection looks like this (note `Is mcp connection: true` and that the token itself is not shown):

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_connection_detail.png" width="70%">

### 2c. Register it as an MCP Service in the AI Gateway

1. Go to **AI Gateway** (left nav, under AI/ML) and open the **MCPs** tab. This is where managed and external MCP servers are governed together.

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_mcp_services_list.png" width="70%">

2. Click **+ MCP**, choose the `weather_mcp_conn` connection you just made, give the service a name (`weather_mcp`) in a schema you own, and create it. You can optionally restrict which of the server's tools are exposed (**tool filtering**).
3. Open the new service. The detail page shows its tools, a **Get started** panel with the proxy URL, and tabs for **Metrics**, **Permissions**, and **Policies** — the governance surface.

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_mcp_service_detail.png" width="70%">

The gateway proxy URL is `https://<workspace-host>/ai-gateway/mcp-services/<catalog>.<schema>.weather_mcp`.

### 2d. Grant access

On the service's **Permissions** tab, click **Grant**, pick the group or service principal that should use the tool, and give it **EXECUTE**. One grant covers all of the service's tools.

> **Do not** grant `USE CONNECTION` on the underlying connection to end users — that lets them bypass the gateway (and its tool filtering, policies, and audit log). `EXECUTE` on the MCP Service is all they need.

## 3. Use the governed tool in an agent

Try it in the **AI Playground** first (left nav → Playground):

1. Click **Tools → + Add tool**, then the **MCP Servers** tab. Notice this one panel contains all three MCP types from the start of the notebook:

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_playground_add_mcp_tool.png" width="70%">

2. Under **UC MCP Service** — the one that says *"Requests are routed through AI Gateway"* — select `weather_mcp`. Click **Save**.
3. Ask the model something like *"What's the current weather in Denver?"* and watch it call the tool. (Behind the scenes the call goes through the gateway.)

### Add it to a Multi-Agent Supervisor

To put it in the telecom supervisor from `05_Multi_Supervisor_Agent`: go to **Agents**, open your supervisor (or **Create Agent → Multi-Agent Supervisor**).

<img src="https://raw.githubusercontent.com/chen-data-ai/Agent-Bricks-Workshop/refs/heads/main/resources/screenshots/screenshot_gw_agents_page.png" width="70%">

Add a tool of type **Unity Catalog MCP Service** and select `weather_mcp`. Name it `Weather_Conditions` and describe when to use it (current conditions / forecasts for field-technician dispatch and outage response), with a couple of example prompts. Save, and the supervisor rebuilds.

## 4. The payoff: it's governed

Because the tool is a gateway-registered MCP Service, you get:

- **Permissions** — only principals with `EXECUTE` can call it; nobody got `USE CONNECTION`, so the gateway can't be bypassed.
- **Tool filtering** — expose only the tools you want (set when you registered the service).
- **Service policies** — the **Policies** tab lets you allow / deny / require approval on specific tool calls.
- **Audit trail** — every call is logged. In a SQL query:

```sql
SELECT event_time, user_identity.email, service_name, action_name
FROM system.access.audit
WHERE event_date >= current_date() - 1
  AND (lower(action_name) LIKE '%mcp%' OR lower(service_name) LIKE '%mcp%')
ORDER BY event_time DESC
LIMIT 50
```

Usage and latency for the service also appear on its **Metrics** tab and in the AI Gateway usage view.

## Recap

You did the whole MCP-through-the-gateway flow in the console: created the HTTP connection, registered the MCP Service, granted `EXECUTE`, and used the governed weather tool in an agent. For the code/SDK version of these same steps, see `02_MCP on Databricks`.

### Learn more
- [Register an external MCP server (Unity AI Gateway)](https://docs.databricks.com/aws/en/ai-gateway/register-mcp-service)
- [Govern MCP Services](https://docs.databricks.com/aws/en/ai-gateway/govern-mcp-service)
- [Custom MCP servers on Databricks Apps](https://docs.databricks.com/aws/en/generative-ai/mcp/custom-mcp)